In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Add project root to path
PROJECT_ROOT = Path(r"c:\Users\nikhi\Desktop\CXR_LT_ISBI")
sys.path.insert(0, str(PROJECT_ROOT))

# Import our modules
import config
from dataloader import create_dataloaders, CXRDataset
from models import (
    get_model, MODEL_REGISTRY,
    DenseNet121Classifier, EfficientNetB4Classifier, ConvNeXtTinyClassifier,
    CXRDenseNet121, CXRResNet50,
    MLGCN, MLGCNWithAttention, build_adjacency_matrix
)
from losses import LDAMDRWLoss, AsymmetricLoss, FocalLoss
from trainer import train_model, train_two_stage, validate, predict
from metrics import compute_metrics
from utils import set_seed, create_submission
from ensemble import (
    ensemble_predict, average_predictions, weighted_average_predictions,
    TTAWrapper, optimize_ensemble_weights
)

set_seed(config.SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Show available models
print("Available models in registry:")
for name in MODEL_REGISTRY.keys():
    print(f"  - {name}")

## Setup: Load Data and Compute Class Statistics

In [ ]:
# Load training data
train_df = pd.read_csv(config.TRAIN_CSV)
print(f"Training samples: {len(train_df)}")
print(f"Number of classes: {len(config.CLASS_NAMES)}")

# Compute class counts for losses
class_counts = train_df[config.CLASS_NAMES].sum().values
total_samples = len(train_df)

print("\nClass distribution (top 10 and bottom 10):")
sorted_idx = np.argsort(class_counts)[::-1]
for i in sorted_idx[:5]:
    print(f"  {config.CLASS_NAMES[i]}: {class_counts[i]:,}")
print("  ...")
for i in sorted_idx[-5:]:
    print(f"  {config.CLASS_NAMES[i]}: {class_counts[i]:,}")

In [ ]:
# Create dataloaders
train_loader, val_loader = create_dataloaders(
    train_df,
    config.IMAGE_DIR,
    config.CLASS_NAMES,
    batch_size=config.BATCH_SIZE,
    image_size=config.IMAGE_SIZE,
    val_split=0.1,
    num_workers=config.NUM_WORKERS
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

---
# OPTION 1: Multiple Backbone Training

Train different backbones for later ensemble.

In [ ]:
# Select backbone to train
BACKBONE_CHOICE = "densenet121"  # Change to: "efficientnet_b4", "convnext_tiny", etc.

model = get_model(
    model_name=BACKBONE_CHOICE,
    num_classes=len(config.CLASS_NAMES),
    pretrained=True,
    dropout=0.5
)
model = model.to(device)

print(f"Model: {BACKBONE_CHOICE}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Setup training with LDAM+DRW loss (best for long-tailed)
from losses import LDAMDRWLoss

criterion = LDAMDRWLoss(
    class_counts=class_counts,
    max_m=0.5,
    s=30,
    drw_start_epoch=5
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.NUM_EPOCHS,
    eta_min=1e-6
)

In [ ]:
# Train
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=config.NUM_EPOCHS,
    device=device,
    save_path=PROJECT_ROOT / f"checkpoints/{BACKBONE_CHOICE}_ldam_drw_best.pth"
)

---
# OPTION 2: CXR Pre-trained Weights

Use models pre-trained on CXR data (CheXpert, MIMIC-CXR) for better initialization.

In [ ]:
# Option 2A: Using custom pretrained weights (download first)
# Example: Download CheXpert pretrained DenseNet121 weights

# Set path to your downloaded weights
PRETRAINED_WEIGHTS_PATH = None  # e.g., "checkpoints/chexpert_densenet121.pth"

if PRETRAINED_WEIGHTS_PATH:
    model = CXRDenseNet121(
        num_classes=len(config.CLASS_NAMES),
        pretrained=True,
        pretrained_path=PRETRAINED_WEIGHTS_PATH,
        pretrained_source="chexpert",
        dropout=0.5,
        freeze_backbone=False  # Fine-tune all layers
    )
else:
    # Fallback to ImageNet pretrained
    print("No CXR pretrained weights specified, using ImageNet pretrained")
    model = CXRDenseNet121(
        num_classes=len(config.CLASS_NAMES),
        pretrained=True,
        pretrained_source="imagenet",
        dropout=0.5
    )

model = model.to(device)

In [ ]:
# Option 2B: Using TorchXRayVision (install first: pip install torchxrayvision)
# This provides DenseNet121 trained on NIH + CheXpert + MIMIC + PadChest + more

USE_TORCHXRAYVISION = False  # Set to True if you have it installed

if USE_TORCHXRAYVISION:
    try:
        from models.cxr_pretrained import TorchXRayVisionWrapper
        
        model = TorchXRayVisionWrapper(
            num_classes=len(config.CLASS_NAMES),
            weights="densenet121-res512-all",  # Pre-trained on multiple CXR datasets
            dropout=0.5,
            freeze_backbone=True  # Start with frozen backbone
        )
        model = model.to(device)
        print("Using TorchXRayVision pretrained model")
    except ImportError as e:
        print(f"TorchXRayVision not available: {e}")
        print("Install with: pip install torchxrayvision")

In [ ]:
# Two-stage training for CXR pretrained:
# Stage 1: Train only classifier head (backbone frozen)
# Stage 2: Fine-tune entire model with lower LR

# Note: If using TorchXRayVision with freeze_backbone=True, 
# you can use train_two_stage for better results

criterion = LDAMDRWLoss(
    class_counts=class_counts,
    max_m=0.5,
    s=30,
    drw_start_epoch=3
)

history = train_two_stage(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    device=device,
    stage1_epochs=5,
    stage2_epochs=15,
    stage1_lr=1e-3,
    stage2_lr=1e-5,
    save_path=PROJECT_ROOT / "checkpoints/cxr_pretrained_best.pth"
)

---
# OPTION 3: ML-GCN for Label Co-occurrence

Uses Graph Convolutional Networks to model relationships between labels.

In [ ]:
# Build adjacency matrix from training data
# This captures how often labels co-occur

train_labels = train_df[config.CLASS_NAMES].values
adj_matrix = build_adjacency_matrix(train_labels, len(config.CLASS_NAMES))

print("Adjacency matrix shape:", adj_matrix.shape)
print("\nExample co-occurrence probabilities:")
for i in range(min(5, len(config.CLASS_NAMES))):
    top_3_idx = np.argsort(adj_matrix[i])[-4:-1][::-1]  # Top 3 excluding self
    print(f"  {config.CLASS_NAMES[i]}:")
    for j in top_3_idx:
        if i != j:
            print(f"    -> {config.CLASS_NAMES[j]}: {adj_matrix[i,j]:.3f}")

In [ ]:
# Create ML-GCN model
model = MLGCN(
    num_classes=len(config.CLASS_NAMES),
    backbone="resnet50",
    pretrained=True,
    adj_matrix=adj_matrix,
    embed_dim=300,
    hidden_dim=1024,
    dropout=0.5,
    t=0.4  # Co-occurrence threshold
)
model = model.to(device)

print(f"ML-GCN model created")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Alternative: ML-GCN with Attention
# Uses attention to weight spatial features based on label embeddings

USE_ATTENTION_VARIANT = False

if USE_ATTENTION_VARIANT:
    model = MLGCNWithAttention(
        num_classes=len(config.CLASS_NAMES),
        backbone="resnet50",
        pretrained=True,
        adj_matrix=adj_matrix,
        embed_dim=300,
        hidden_dim=1024,
        dropout=0.5
    )
    model = model.to(device)
    print("Using ML-GCN with Attention")

In [ ]:
# Train ML-GCN
criterion = AsymmetricLoss(gamma_pos=0, gamma_neg=4, clip=0.05)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.NUM_EPOCHS,
    eta_min=1e-6
)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=config.NUM_EPOCHS,
    device=device,
    save_path=PROJECT_ROOT / "checkpoints/mlgcn_best.pth"
)

---
# OPTION 4: Ensemble Training

Train multiple models and combine their predictions.

**Strategy**: Train 3-5 diverse models, then ensemble.

In [ ]:
# Define models to train for ensemble
ENSEMBLE_CONFIGS = [
    {"name": "densenet121", "backbone": "densenet121", "loss": "ldam_drw"},
    {"name": "efficientnet_b4", "backbone": "efficientnet_b4", "loss": "asymmetric"},
    {"name": "convnext_tiny", "backbone": "convnext_tiny", "loss": "ldam_drw"},
]

print("Models to train for ensemble:")
for cfg in ENSEMBLE_CONFIGS:
    print(f"  - {cfg['name']} with {cfg['loss']} loss")

In [ ]:
# Function to train a single model for ensemble
def train_for_ensemble(config_dict, train_loader, val_loader, class_counts, device):
    """Train a single model configuration."""
    
    name = config_dict["name"]
    backbone = config_dict["backbone"]
    loss_type = config_dict["loss"]
    
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")
    
    # Create model
    model = get_model(
        model_name=backbone,
        num_classes=len(config.CLASS_NAMES),
        pretrained=True
    ).to(device)
    
    # Create loss
    if loss_type == "ldam_drw":
        criterion = LDAMDRWLoss(class_counts, max_m=0.5, s=30, drw_start_epoch=5)
    elif loss_type == "asymmetric":
        criterion = AsymmetricLoss(gamma_pos=0, gamma_neg=4, clip=0.05)
    else:
        criterion = FocalLoss(alpha=1.0, gamma=2.0)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
    
    # Train
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=15,
        device=device,
        save_path=PROJECT_ROOT / f"checkpoints/ensemble_{name}_best.pth"
    )
    
    return model, history

In [ ]:
# Train all models (this will take a while!)
# Uncomment to run

# trained_models = []
# for cfg in ENSEMBLE_CONFIGS:
#     model, history = train_for_ensemble(
#         cfg, train_loader, val_loader, class_counts, device
#     )
#     trained_models.append(model)

In [ ]:
# Load pre-trained ensemble models (if already trained)
from models import DenseNet121Classifier, EfficientNetB4Classifier, ConvNeXtTinyClassifier

# Example of loading trained models
checkpoint_dir = PROJECT_ROOT / "checkpoints"

def load_ensemble_models(checkpoint_paths, model_classes, num_classes, device):
    """Load multiple trained models."""
    models = []
    for path, model_class in zip(checkpoint_paths, model_classes):
        if Path(path).exists():
            model = model_class(num_classes=num_classes, pretrained=False)
            checkpoint = torch.load(path, map_location=device)
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            model.to(device)
            model.eval()
            models.append(model)
            print(f"Loaded: {path}")
        else:
            print(f"Not found: {path}")
    return models

# Uncomment to load models:
# ensemble_models = load_ensemble_models(
#     checkpoint_paths=[
#         checkpoint_dir / "ensemble_densenet121_best.pth",
#         checkpoint_dir / "ensemble_efficientnet_b4_best.pth",
#         checkpoint_dir / "ensemble_convnext_tiny_best.pth",
#     ],
#     model_classes=[DenseNet121Classifier, EfficientNetB4Classifier, ConvNeXtTinyClassifier],
#     num_classes=len(config.CLASS_NAMES),
#     device=device
# )

---
# Ensemble Prediction & Submission

In [ ]:
# Create test dataloader
test_df = pd.read_csv(config.TEST_CSV)
print(f"Test samples: {len(test_df)}")

from dataloader import get_val_transforms
from torch.utils.data import DataLoader

test_dataset = CXRDataset(
    df=test_df,
    image_dir=config.IMAGE_DIR,
    class_names=config.CLASS_NAMES,
    transform=get_val_transforms(config.IMAGE_SIZE),
    is_test=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

In [ ]:
# Simple prediction with single model
# Use your best trained model here

# Example with a single model:
# model = DenseNet121Classifier(num_classes=len(config.CLASS_NAMES), pretrained=False)
# model.load_state_dict(torch.load(checkpoint_dir / "densenet121_ldam_drw_best.pth")["model_state_dict"])
# model = model.to(device)
# 
# predictions = predict(model, test_loader, device)
# submission = create_submission(test_df, predictions, config.CLASS_NAMES)
# submission.to_csv(PROJECT_ROOT / "submissions/single_model_submission.csv", index=False)

In [ ]:
# Ensemble prediction with TTA

# Assuming ensemble_models is loaded
# predictions = ensemble_predict(
#     models=ensemble_models,
#     dataloader=test_loader,
#     device=device,
#     weights=None,  # Equal weights, or optimize with validation set
#     use_tta=True
# )
# 
# submission = create_submission(test_df, predictions, config.CLASS_NAMES)
# submission.to_csv(PROJECT_ROOT / "submissions/ensemble_tta_submission.csv", index=False)
# print("Submission saved!")

In [ ]:
# Optimize ensemble weights using validation set

# Get validation predictions from each model
# val_predictions = []
# for model in ensemble_models:
#     preds = predict(model, val_loader, device)
#     val_predictions.append(preds)

# Get validation labels
# val_labels = val_dataset.df[config.CLASS_NAMES].values

# Define metric function
# def mAP_metric(preds, labels):
#     from sklearn.metrics import average_precision_score
#     return average_precision_score(labels, preds, average='macro')

# Optimize weights
# best_weights, best_score = optimize_ensemble_weights(
#     val_predictions, val_labels, mAP_metric, num_iterations=1000
# )
# print(f"\nOptimal weights: {best_weights}")
# print(f"Best validation mAP: {best_score:.4f}")

---
# Summary

Phase 3 provides:

1. **Multiple Backbones**: DenseNet121, EfficientNet-B4, ConvNeXt-Tiny
2. **CXR Pretrained**: Load weights from CheXpert/MIMIC/TorchXRayVision
3. **ML-GCN**: Model label co-occurrence with graph convolutions
4. **Ensemble**: Combine multiple models for better robustness

## Recommended Training Strategy:

1. Train 3-5 diverse models (different backbones + loss functions)
2. Use CXR pretrained weights if available (especially TorchXRayVision)
3. Include ML-GCN for label relationship modeling
4. Ensemble with optimized weights + TTA

Expected improvement: 2-5% mAP over single model baseline.